# Addressing Action Diversity: Train CQL & BCQ on Lift

**Date:** 2026-03-17
**Goal:** Train offline RL policies (CQL, BCQ) on Lift to produce target policies with genuinely different action distributions from BC.

**Motivation:** Diagnostic 3 showed +72% GD convergence for BC policies (all agree on actions) vs -5.9% divergence on Hopper (policies disagree). We need policies trained with different objectives (RL vs BC) to get the divergence signal needed for guidance-based OPE.

**Plan:**
1. Train CQL on Lift (full 200 demos)
2. Train BCQ on Lift (full 200 demos)
3. Run Diagnostic 3 (gradient direction test) on {BC_10d, BC_200d, CQL, BCQ, random}
4. Run Diagnostic 1 (cosine similarity of scorer gradients)
5. Compare cross-policy action diversity metrics

In [ ]:
import sys, os, json, time, shutil
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import numpy as np
import torch
import h5py

# Add paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "third_party/robomimic"))

import robomimic.utils.torch_utils as TorchUtils
from robomimic.config import config_factory
from robomimic.scripts.train import train

DEVICE = TorchUtils.get_torch_device(try_to_use_cuda=True)
print(f"Device: {DEVICE}")

DATASET_PATH = os.path.join(PROJECT_ROOT, "third_party/robomimic/datasets/lift/ph/low_dim_v15.hdf5")
OUTPUT_BASE = os.path.join(PROJECT_ROOT, "third_party/robomimic/diffusion_policy_trained_models")
assert os.path.exists(DATASET_PATH), f"Dataset not found: {DATASET_PATH}"
print(f"Dataset: {DATASET_PATH}")
print(f"Output base: {OUTPUT_BASE}")

## Part 1: Train CQL on Lift (200 demos)

In [ ]:
# Build CQL config
cql_cfg = config_factory("cql")

with cql_cfg.values_unlocked():
    # Data
    cql_cfg.train.data = [{"path": DATASET_PATH}]
    cql_cfg.train.output_dir = os.path.join(OUTPUT_BASE, "lift_cql_200demos")
    cql_cfg.train.num_epochs = 200
    cql_cfg.train.batch_size = 1024
    cql_cfg.train.seed = 42
    cql_cfg.train.hdf5_cache_mode = "all"
    cql_cfg.train.hdf5_load_next_obs = True
    
    # Experiment
    cql_cfg.experiment.name = "lift_cql_200demos"
    cql_cfg.experiment.save.every_n_epochs = 50
    cql_cfg.experiment.save.on_best_rollout_success_rate = True
    cql_cfg.experiment.save.epochs = [10, 25, 50, 100, 150, 200]
    
    # Rollout eval every 25 epochs, 20 rollouts (faster than default 50)
    cql_cfg.experiment.rollout.enabled = True
    cql_cfg.experiment.rollout.n = 20
    cql_cfg.experiment.rollout.horizon = 400
    cql_cfg.experiment.rollout.rate = 25
    cql_cfg.experiment.rollout.terminate_on_success = True
    
    # No video rendering to save time
    cql_cfg.experiment.render_video = False
    
    # Logging
    cql_cfg.experiment.logging.log_wandb = False
    cql_cfg.experiment.logging.terminal_output_to_txt = False  # keep output in notebook

cql_cfg.lock()
print("CQL config created.")
print(f"  Epochs: {cql_cfg.train.num_epochs}")
print(f"  Batch size: {cql_cfg.train.batch_size}")
print(f"  Output: {cql_cfg.train.output_dir}")

In [ ]:
# Delete existing output dirs to avoid robomimic's interactive overwrite prompt
cql_out = cql_cfg.train.output_dir
if os.path.exists(cql_out):
    shutil.rmtree(cql_out)
    print(f"Removed existing CQL output dir: {cql_out}")

print("="*60)
print("TRAINING CQL ON LIFT (200 demos, 200 epochs)")
print("="*60)
t0 = time.time()
train(cql_cfg, device=DEVICE)
cql_time = time.time() - t0
print(f"\nCQL training completed in {cql_time/60:.1f} minutes")

In [ ]:
# Find the CQL checkpoint that was just saved
import glob

cql_output_dir = cql_cfg.train.output_dir
cql_run_dirs = sorted(glob.glob(os.path.join(cql_output_dir, "**", "models"), recursive=True), key=os.path.getmtime)
assert len(cql_run_dirs) > 0, f"No CQL checkpoints found in {cql_output_dir}"
cql_models_dir = cql_run_dirs[-1]
cql_run_dir = os.path.dirname(cql_models_dir)
cql_ckpts = sorted(glob.glob(os.path.join(cql_models_dir, "model_epoch_*.pth")))
print(f"CQL run dir: {cql_run_dir}")
print(f"CQL checkpoints saved: {len(cql_ckpts)}")
for c in cql_ckpts:
    print(f"  {os.path.basename(c)}")

## Part 2: Train BCQ on Lift (200 demos)

In [ ]:
# Build BCQ config
bcq_cfg = config_factory("bcq")

with bcq_cfg.values_unlocked():
    # Data
    bcq_cfg.train.data = [{"path": DATASET_PATH}]
    bcq_cfg.train.output_dir = os.path.join(OUTPUT_BASE, "lift_bcq_200demos")
    bcq_cfg.train.num_epochs = 200
    bcq_cfg.train.batch_size = 256
    bcq_cfg.train.seed = 42
    bcq_cfg.train.hdf5_cache_mode = "all"
    bcq_cfg.train.hdf5_load_next_obs = True
    
    # Experiment
    bcq_cfg.experiment.name = "lift_bcq_200demos"
    bcq_cfg.experiment.save.every_n_epochs = 50
    bcq_cfg.experiment.save.on_best_rollout_success_rate = True
    bcq_cfg.experiment.save.epochs = [10, 25, 50, 100, 150, 200]
    
    # Rollout eval
    bcq_cfg.experiment.rollout.enabled = True
    bcq_cfg.experiment.rollout.n = 20
    bcq_cfg.experiment.rollout.horizon = 400
    bcq_cfg.experiment.rollout.rate = 25
    bcq_cfg.experiment.rollout.terminate_on_success = True
    
    # No video rendering
    bcq_cfg.experiment.render_video = False
    
    # Logging
    bcq_cfg.experiment.logging.log_wandb = False
    bcq_cfg.experiment.logging.terminal_output_to_txt = False

bcq_cfg.lock()
print("BCQ config created.")
print(f"  Epochs: {bcq_cfg.train.num_epochs}")
print(f"  Batch size: {bcq_cfg.train.batch_size}")
print(f"  Output: {bcq_cfg.train.output_dir}")

In [ ]:
# Delete existing output dirs to avoid robomimic's interactive overwrite prompt
bcq_out = bcq_cfg.train.output_dir
if os.path.exists(bcq_out):
    shutil.rmtree(bcq_out)
    print(f"Removed existing BCQ output dir: {bcq_out}")

print("="*60)
print("TRAINING BCQ ON LIFT (200 demos, 200 epochs)")
print("="*60)
t0 = time.time()
train(bcq_cfg, device=DEVICE)
bcq_time = time.time() - t0
print(f"\nBCQ training completed in {bcq_time/60:.1f} minutes")

In [ ]:
# Find the BCQ checkpoint
bcq_output_dir = bcq_cfg.train.output_dir
bcq_run_dirs = sorted(glob.glob(os.path.join(bcq_output_dir, "**", "models"), recursive=True), key=os.path.getmtime)
assert len(bcq_run_dirs) > 0, f"No BCQ checkpoints found in {bcq_output_dir}"
bcq_models_dir = bcq_run_dirs[-1]
bcq_run_dir = os.path.dirname(bcq_models_dir)
bcq_ckpts = sorted(glob.glob(os.path.join(bcq_models_dir, "model_epoch_*.pth")))
print(f"BCQ run dir: {bcq_run_dir}")
print(f"BCQ checkpoints saved: {len(bcq_ckpts)}")
for c in bcq_ckpts:
    print(f"  {os.path.basename(c)}")

## Part 3: Load All Policies for Diversity Diagnostics

Policy set: {BC_10demos, BC_200demos, CQL, BCQ, Random}

In [ ]:
from src.latent_sope.robomimic_interface.checkpoints import (
    load_checkpoint, build_algo_from_checkpoint, build_rollout_policy_from_checkpoint
)
import robomimic.utils.obs_utils as ObsUtils
from robomimic.algo import algo_factory, RolloutPolicy
import robomimic.utils.file_utils as FileUtils

%matplotlib inline
import matplotlib.pyplot as plt

OBS_KEYS = ["object", "robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos"]
STATE_DIM = 19  # 10 + 3 + 4 + 2
ACTION_DIM = 7

# --- Load BC policies ---
CKPT_BASE = os.path.join(PROJECT_ROOT, "third_party/robomimic/diffusion_policy_trained_models")

bc_configs = {
    "BC_10d": (f"{CKPT_BASE}/lift_diffusion_10demos/20260311115828", "models/model_epoch_40.pth"),
    "BC_200d": (f"{CKPT_BASE}/lift_diffusion_200demos/20260311141036", "models/model_epoch_40.pth"),
}

rollout_policies = {}
algo_models = {}  # raw algo objects for gradient access

for name, (run_dir, ckpt_rel) in bc_configs.items():
    print(f"Loading {name} from {run_dir}...")
    ckpt = load_checkpoint(run_dir=run_dir, ckpt_path=ckpt_rel)
    rollout_policies[name] = build_rollout_policy_from_checkpoint(ckpt, device=DEVICE, verbose=False)
    algo_models[name] = build_algo_from_checkpoint(ckpt, device=DEVICE)
    print(f"  Loaded {name}")

# --- Load CQL ---
print(f"\nLoading CQL from {cql_run_dir}...")
cql_ckpt = load_checkpoint(run_dir=cql_run_dir)
rollout_policies["CQL"] = build_rollout_policy_from_checkpoint(cql_ckpt, device=DEVICE, verbose=False)
algo_models["CQL"] = build_algo_from_checkpoint(cql_ckpt, device=DEVICE)
print(f"  Loaded CQL (epoch {cql_ckpt.epoch})")

# --- Load BCQ ---
print(f"\nLoading BCQ from {bcq_run_dir}...")
bcq_ckpt = load_checkpoint(run_dir=bcq_run_dir)
rollout_policies["BCQ"] = build_rollout_policy_from_checkpoint(bcq_ckpt, device=DEVICE, verbose=False)
algo_models["BCQ"] = build_algo_from_checkpoint(bcq_ckpt, device=DEVICE)
print(f"  Loaded BCQ (epoch {bcq_ckpt.epoch})")

print(f"\nLoaded {len(rollout_policies)} policies: {list(rollout_policies.keys())}")

In [ ]:
# Load demo data for evaluation states
N_STATES = 500
np.random.seed(42)
torch.manual_seed(42)

with h5py.File(DATASET_PATH, "r") as f:
    demo_keys = sorted(list(f["data"].keys()))
    all_obs = {k: [] for k in OBS_KEYS}
    all_actions = []
    for dk in demo_keys:
        for k in OBS_KEYS:
            all_obs[k].append(f["data"][dk]["obs"][k][:])
        all_actions.append(f["data"][dk]["actions"][:])
    for k in OBS_KEYS:
        all_obs[k] = np.concatenate(all_obs[k], axis=0)
    all_actions_np = np.concatenate(all_actions, axis=0)

total_steps = all_obs[OBS_KEYS[0]].shape[0]
action_std = np.std(all_actions_np, axis=0)
print(f"Dataset: {total_steps} transitions, action_std = {action_std}")

# Sample random states
sample_idx = np.random.choice(total_steps, N_STATES, replace=False)
sampled_obs_dicts = [{k: all_obs[k][i] for k in OBS_KEYS} for i in sample_idx]
sampled_actions = all_actions_np[sample_idx]

# Also build concatenated state vectors
sampled_states = np.concatenate([all_obs[k][sample_idx] for k in OBS_KEYS], axis=-1)
print(f"Sampled {N_STATES} states, state shape: {sampled_states.shape}")

## Part 4: Cross-Policy Action Queries

In [ ]:
# Query each policy at sampled states
# For BC (DiffusionPolicyUNet): use rollout policy with stacked obs
# For CQL/BCQ: use rollout policy

policy_actions = {}  # {name: (N_STATES, ACTION_DIM)}

for name, policy in sorted(rollout_policies.items()):
    print(f"Querying {name}...")
    actions_list = []
    for i, obs_dict in enumerate(sampled_obs_dicts):
        policy.start_episode()
        # Stack obs for policies that need observation_horizon > 1
        obs_input = {k: np.stack([v, v], axis=0).astype(np.float32) for k, v in obs_dict.items()}
        act = policy(obs_input)
        actions_list.append(act)
        if (i + 1) % 100 == 0:
            print(f"  {i+1}/{N_STATES}")
    policy_actions[name] = np.stack(actions_list, axis=0)
    print(f"  {name}: mean={policy_actions[name].mean(0)}, std={policy_actions[name].std(0)}")

# Add random policy
random_actions = np.random.uniform(-1, 1, size=(N_STATES, ACTION_DIM))
policy_actions["Random"] = random_actions
print(f"\nCollected actions from {len(policy_actions)} policies")

## Part 5: Cross-Policy Action Diversity Metrics

In [ ]:
from itertools import combinations

def compute_diversity_metrics(policy_actions, action_std, domain_name):
    names = sorted(policy_actions.keys())
    N = next(iter(policy_actions.values())).shape[0]
    action_dim = next(iter(policy_actions.values())).shape[1]
    n_policies = len(names)
    stacked = np.stack([policy_actions[n] for n in names], axis=0)  # (n_policies, N, action_dim)
    
    # Metric 1: Per-state cross-policy action std (normalized)
    per_state_std = np.std(stacked, axis=0)
    normalized_std = per_state_std / action_std[None, :]
    mean_normalized_std = np.mean(normalized_std)
    per_dim_normalized_std = np.mean(normalized_std, axis=0)
    
    # Metric 2: Pairwise mean L2 distance (normalized)
    pairwise_dists = {}
    for i, j in combinations(range(n_policies), 2):
        diff = (stacked[i] - stacked[j]) / action_std[None, :]
        l2 = np.sqrt(np.sum(diff**2, axis=-1))
        pairwise_dists[(names[i], names[j])] = np.mean(l2)
    mean_pairwise_dist = np.mean(list(pairwise_dists.values()))
    
    # Metric 3: Agreement rate
    threshold = 0.1
    per_state_range = np.max(stacked, axis=0) - np.min(stacked, axis=0)
    normalized_range = per_state_range / action_std[None, :]
    all_agree = np.all(normalized_range < threshold, axis=-1)
    agreement_rate = np.mean(all_agree)
    
    # Print
    print(f"\n{'='*60}")
    print(f"{domain_name} Action Diversity Metrics")
    print(f"{'='*60}")
    print(f"Policies: {n_policies} — {names}")
    print(f"States: {N}, Action dim: {action_dim}")
    print(f"")
    print(f"Mean normalized cross-policy std: {mean_normalized_std:.4f}")
    print(f"  Per-dim: {per_dim_normalized_std}")
    print(f"Mean pairwise L2 distance (normalized): {mean_pairwise_dist:.4f}")
    print(f"Agreement rate (all within 0.1σ): {agreement_rate*100:.1f}%")
    print(f"")
    print(f"Pairwise distances:")
    for (ni, nj), d in sorted(pairwise_dists.items()):
        print(f"  {ni:>10s} vs {nj:<10s}: {d:.4f}")
    
    return {
        "mean_normalized_std": mean_normalized_std,
        "per_dim_normalized_std": per_dim_normalized_std,
        "mean_pairwise_dist": mean_pairwise_dist,
        "pairwise_dists": pairwise_dists,
        "agreement_rate": agreement_rate,
        "names": names,
    }

# With random
results_all = compute_diversity_metrics(policy_actions, action_std, "Lift (BC + CQL + BCQ + Random)")

# Without random (to see diversity among learned policies only)
learned_actions = {k: v for k, v in policy_actions.items() if k != "Random"}
results_learned = compute_diversity_metrics(learned_actions, action_std, "Lift (BC + CQL + BCQ only)")

In [ ]:
# Pairwise distance heatmap
names = results_all["names"]
n = len(names)
pw_matrix = np.zeros((n, n))
for (ni, nj), d in results_all["pairwise_dists"].items():
    i, j = names.index(ni), names.index(nj)
    pw_matrix[i, j] = d
    pw_matrix[j, i] = d

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
im = ax.imshow(pw_matrix, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_yticklabels(names)
ax.set_title("Pairwise L2 Distance (normalized by action std)")
plt.colorbar(im, ax=ax)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{pw_matrix[i,j]:.2f}", ha='center', va='center', fontsize=10)
plt.tight_layout()
plt.show()

## Part 6: Diagnostic 3 — Gradient Direction Test

For each policy, start from random actions and run gradient ascent on the policy's scoring function.
Measure whether actions converge toward (+%) or diverge from (-%) dataset actions.

- **CQL** has a Gaussian actor — we can extract mean/std for analytic grad_log_prob.
- **BCQ** uses a VAE — we use finite-difference gradient estimation on reconstructed log-prob.
- **BC (DiffusionPolicyUNet)** — we use the rollout policy to estimate gradients via finite differences.

In [ ]:
# --- Scoring functions per policy type ---

def cql_grad_log_prob(algo_model, obs_dict_batch, actions_batch):
    """
    CQL has a Gaussian actor. Extract mean/std and compute analytic gradient.
    grad log p(a|s) = -(a - mean) / std^2
    """
    obs_torch = {k: torch.tensor(v, dtype=torch.float32, device=DEVICE) for k, v in obs_dict_batch.items()}
    actions_torch = torch.tensor(actions_batch, dtype=torch.float32, device=DEVICE).requires_grad_(True)
    
    # Get actor distribution
    actor = algo_model.nets["actor"]
    dist = actor.forward_train(obs_dict=obs_torch)
    log_prob = dist.log_prob(actions_torch).sum(-1)  # (B,)
    log_prob.sum().backward()
    grad = actions_torch.grad.detach().cpu().numpy()
    return grad


def finite_diff_grad(policy, obs_dict, action, eps=1e-3):
    """
    Finite-difference gradient for policies without analytic grad_log_prob.
    Uses L2 distance to policy's preferred action as a proxy score.
    """
    # Get policy's preferred action at this state
    policy.start_episode()
    obs_stacked = {k: np.stack([v, v], axis=0).astype(np.float32) for k, v in obs_dict.items()}
    preferred = policy(obs_stacked)
    
    # Gradient of -0.5 * ||a - preferred||^2 = -(a - preferred) = preferred - a
    return preferred - action


def run_gd_test(policy_name, grad_fn, obs_dicts, dataset_actions, n_states=200, n_steps=50, lr=0.1):
    """
    Gradient direction test: start from random actions, ascend scoring function,
    measure L2 distance to dataset actions before and after.
    """
    improvements = []
    for i in range(n_states):
        # Random starting action
        action = np.random.randn(ACTION_DIM) * 0.5
        target = dataset_actions[i]
        
        init_dist = np.linalg.norm(action - target)
        
        # Gradient ascent
        for step in range(n_steps):
            grad = grad_fn(obs_dicts[i], action)
            action = action + lr * grad
            action = np.clip(action, -1, 1)
        
        final_dist = np.linalg.norm(action - target)
        improvement = (init_dist - final_dist) / max(init_dist, 1e-8) * 100
        improvements.append(improvement)
    
    mean_imp = np.mean(improvements)
    std_imp = np.std(improvements)
    return mean_imp, std_imp, improvements

print("Diagnostic functions defined.")

In [ ]:
# Run Diagnostic 3 for each policy
N_GD_STATES = 200
gd_results = {}

for name in sorted(policy_actions.keys()):
    if name == "Random":
        print(f"Skipping {name} (no scoring function)")
        continue
    
    print(f"\nRunning GD test for {name}...")
    
    if name == "CQL":
        # CQL: use analytic gradient via Gaussian actor
        def grad_fn(obs_dict, action):
            obs_batch = {k: np.expand_dims(v, 0) for k, v in obs_dict.items()}
            act_batch = np.expand_dims(action, 0)
            return cql_grad_log_prob(algo_models["CQL"], obs_batch, act_batch)[0]
    else:
        # BC and BCQ: use finite-difference via rollout policy preferred action
        policy = rollout_policies[name]
        def grad_fn(obs_dict, action, _p=policy):
            return finite_diff_grad(_p, obs_dict, action)
    
    mean_imp, std_imp, improvements = run_gd_test(
        name, grad_fn, sampled_obs_dicts[:N_GD_STATES], sampled_actions[:N_GD_STATES],
        n_states=N_GD_STATES, n_steps=50, lr=0.1
    )
    gd_results[name] = {"mean": mean_imp, "std": std_imp, "raw": improvements}
    sign = "+" if mean_imp > 0 else ""
    print(f"  {name}: {sign}{mean_imp:.1f}% ± {std_imp:.1f}%")

print(f"\n{'='*60}")
print("DIAGNOSTIC 3 SUMMARY: Gradient Direction Test")
print(f"{'='*60}")
print(f"{'Policy':<15s} {'Mean Improvement':>18s} {'Interpretation':>20s}")
print(f"{'-'*55}")
for name, r in sorted(gd_results.items()):
    sign = "+" if r['mean'] > 0 else ""
    interp = "CONVERGE (bad)" if r['mean'] > 10 else ("DIVERGE (good)" if r['mean'] < -2 else "NEUTRAL")
    print(f"{name:<15s} {sign}{r['mean']:>16.1f}% {interp:>20s}")
print(f"\nTarget: negative values (divergence) = policies disagree on actions")
print(f"Reference: Hopper D4RL = -5.9% (good), previous Lift BC-only = +72% (bad)")

In [ ]:
# Diagnostic 3 visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
ax = axes[0]
names_gd = sorted(gd_results.keys())
means = [gd_results[n]["mean"] for n in names_gd]
stds = [gd_results[n]["std"] for n in names_gd]
colors = ["coral" if m > 10 else ("steelblue" if m < -2 else "gray") for m in means]
bars = ax.bar(names_gd, means, yerr=stds, color=colors, alpha=0.8, capsize=5)
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
ax.axhline(y=-5.9, color='green', linestyle=':', linewidth=1, label='Hopper reference (-5.9%)')
ax.set_ylabel("GD Convergence (%)")
ax.set_title("Diagnostic 3: Gradient Direction Test")
ax.legend()
for bar, val in zip(bars, means):
    sign = "+" if val > 0 else ""
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f"{sign}{val:.1f}%", ha='center', va='bottom', fontsize=9)

# Histogram of improvements
ax = axes[1]
for name in names_gd:
    ax.hist(gd_results[name]["raw"], bins=30, alpha=0.5, label=name)
ax.axvline(x=0, color='black', linestyle='--', linewidth=0.5)
ax.set_xlabel("Improvement (%)")
ax.set_ylabel("Count")
ax.set_title("Distribution of Per-State GD Improvements")
ax.legend()

plt.tight_layout()
plt.show()

## Part 7: Rollout Success Rate Comparison

In [ ]:
# Run rollouts for each learned policy to get success rates
from src.latent_sope.robomimic_interface.checkpoints import build_env_from_checkpoint

N_ROLLOUTS = 20
HORIZON = 400

# Build env from any checkpoint (they all use the same env)
env = build_env_from_checkpoint(cql_ckpt, render=False, render_offscreen=False)

sr_results = {}
for name in ["BC_10d", "BC_200d", "CQL", "BCQ"]:
    policy = rollout_policies[name]
    successes = 0
    returns = []
    print(f"Rolling out {name} ({N_ROLLOUTS} episodes)...")
    for ep in range(N_ROLLOUTS):
        obs = env.reset()
        policy.start_episode()
        total_reward = 0
        for t in range(HORIZON):
            act = policy(obs)
            obs, r, done, info = env.step(act)
            total_reward += r
            if done:
                break
        success = env.is_success()["task"]
        successes += int(success)
        returns.append(total_reward)
    sr = successes / N_ROLLOUTS * 100
    sr_results[name] = {"sr": sr, "mean_return": np.mean(returns), "std_return": np.std(returns)}
    print(f"  {name}: SR={sr:.0f}%, Return={np.mean(returns):.1f} ± {np.std(returns):.1f}")

print(f"\n{'='*50}")
print("ROLLOUT SUCCESS RATES")
print(f"{'='*50}")
for name, r in sorted(sr_results.items()):
    print(f"  {name:<10s}: SR={r['sr']:.0f}%, Return={r['mean_return']:.1f} ± {r['std_return']:.1f}")

## Part 8: Final Summary

In [ ]:
print("\n" + "="*70)
print("FINAL SUMMARY: Addressing Action Diversity")
print("="*70)

print(f"\n--- Training ---")
print(f"CQL: {cql_time/60:.1f} min, output: {cql_run_dir}")
print(f"BCQ: {bcq_time/60:.1f} min, output: {bcq_run_dir}")

print(f"\n--- Rollout Success Rates ---")
for name, r in sorted(sr_results.items()):
    print(f"  {name:<10s}: {r['sr']:.0f}%")

print(f"\n--- Action Diversity (learned policies only, no Random) ---")
print(f"Mean normalized cross-policy std: {results_learned['mean_normalized_std']:.4f}")
print(f"Mean pairwise L2 (normalized): {results_learned['mean_pairwise_dist']:.4f}")
print(f"Agreement rate: {results_learned['agreement_rate']*100:.1f}%")

print(f"\n--- Diagnostic 3: Gradient Direction Test ---")
for name, r in sorted(gd_results.items()):
    sign = "+" if r['mean'] > 0 else ""
    print(f"  {name:<15s}: {sign}{r['mean']:.1f}%")
print(f"  Reference (Hopper): -5.9%")
print(f"  Reference (old BC-only Lift): +72%")

print(f"\n--- Verdict ---")
learned_gd = [r['mean'] for name, r in gd_results.items() if name in ["CQL", "BCQ"]]
if any(v < -2 for v in learned_gd):
    print("SUCCESS: At least one RL policy shows divergence — guidance can distinguish policies.")
elif any(v < 10 for v in learned_gd):
    print("PARTIAL: RL policies show reduced convergence but not clear divergence. More training or tuning may help.")
else:
    print("INSUFFICIENT: RL policies still converge like BC. Need fundamentally different approach.")

print(f"\nDone.")